In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc ffsim matplotlib numpy pyscf qiskit-addon-sqd
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Stichprobenbasierte Quantendiagonalisierung eines Chemie-Hamiltonoperators

import TutorialFeedback from '@site/src/components/TutorialFeedback';



*Geschätzter Ressourcenverbrauch: weniger als eine Minute auf einem Heron-r2-Prozessor (HINWEIS: Dies ist nur eine Schätzung. Deine Laufzeit kann abweichen.)*
## Lernziele
Nach dem Durcharbeiten dieses Tutorials sollten die Nutzer/innen Folgendes verstehen:
- Wie man das [SQD-Qiskit-Addon](https://github.com/Qiskit/qiskit-addon-sqd) verwendet, um die Grundzustandsenergie eines molekularen Systems mithilfe von Bitstrings anzunähern, die von einer Quantenverarbeitungseinheit (QPU) abgetastet wurden.
- Wie man [ffsim](https://github.com/qiskit-community/ffsim) verwendet, um einen lokalen unitären Cluster-Jastrow-Ansatz (LUCJ) für die Quantenchemiesimulation zu konstruieren.

## Voraussetzungen
Wir empfehlen, dass du dich mit den folgenden Themen vertraut machst, bevor du dieses Tutorial durcharbeitest:
- Quantenchemie und zweite Quantisierung
- Verwendung des Sampler-Primitives zur Abtastung von Quantenschaltkreisen

## Hintergrund
In diesem Tutorial zeigen wir, wie man verrauschte Quantenstichproben nachbearbeitet, um den Grundzustand des Stickstoffmoleküls $\text{N}_2$ bei der Gleichgewichtsbindungslänge anzunähern, indem man das [SQD-Qiskit-Addon](https://github.com/Qiskit/qiskit-addon-sqd) verwendet, um den [stichprobenbasierten Quantendiagonalisierungsalgorithmus (SQD)](https://arxiv.org/abs/2405.05068) zu implementieren. Weitere Details zur Software sind in der entsprechenden [Dokumentation](/guides/qiskit-addons-sqd) zu finden, einschließlich eines [einfachen Einstiegsbeispiels](/guides/qiskit-addons-sqd-get-started).

Dieses Tutorial richtet sich an Nutzer/innen, die mit der Quantenchemie vertraut sind: insbesondere mit der Bestimmung der Grundzustandsenergien eines Moleküls. Für eine detaillierte Beschreibung des Workflows wird auf den [Kurs zum Quantendiagonalisierungsalgorithmus](https://quantum.cloud.ibm.com/learning/courses/quantum-diagonalization-algorithms) verwiesen.

SQD ist eine Technik zum Auffinden von Eigenwerten und Eigenvektoren von Quantenoperatoren, wie etwa einem Quanten-System-Hamiltonoperator, indem Quanten- und verteiltes klassisches Computing gemeinsam eingesetzt werden. Verteiltes klassisches Computing wird verwendet, um Stichproben zu verarbeiten, die von einem Quantenprozessor gewonnen wurden, und den Ziel-Hamiltonoperator in einem durch die Stichproben aufgespannten Unterraum zu projizieren und zu diagonalisieren. Ein SQD-basierter Workflow hat die folgenden Schritte:

1.  Wähle einen Schaltkreisansatz und wende ihn auf einem Quantencomputer auf einen Referenzzustand an (in diesem Fall den [Hartree-Fock](https://en.wikipedia.org/wiki/Hartree%E2%80%93Fock_method)-Zustand).
2.  Taste Bitstrings aus dem resultierenden Quantenzustand ab.
3.  Führe das Verfahren der *selbstkonsistenten Konfigurationswiederherstellung* auf den Bitstrings aus, um die Grundzustandsnäherung zu erhalten.

Es ist bekannt, dass SQD gut funktioniert, wenn der Ziel-Eigenzustand dünn besetzt ist: Die Wellenfunktion wird in einer Menge von Basiszuständen $\mathcal{S} = {|x\rangle }$ dargestellt, deren Größe nicht exponentiell mit der Problemgröße wächst.

### Quantenchemie
Der Hamiltonoperator eines molekularen Systems lässt sich schreiben als

$$
\hat{H} = \sum_{ \substack{pr\\\sigma} } h_{pr} \, \hat{a}^\dagger_{p\sigma} \hat{a}_{r\sigma}
+ \frac12
\sum_{ \substack{prqs\\\sigma\tau} }
h_{prqs} \,
\hat{a}^\dagger_{p\sigma}
\hat{a}^\dagger_{q\tau}
\hat{a}_{s\tau}
\hat{a}_{r\sigma},
$$

wobei $h_{pr}$ und $h_{prqs}$ komplexe Zahlen sind, die als Molekülintegrale bezeichnet werden und aus der Spezifikation des Moleküls mithilfe eines Computerprogramms berechnet werden können. In diesem Tutorial berechnen wir die Integrale mit dem Softwarepaket [PySCF](https://pyscf.org/).

Für Details zur Herleitung des molekularen Hamiltonoperators konsultiere ein Lehrbuch zur Quantenchemie (zum Beispiel *Modern Quantum Chemistry* von Szabo und Ostlund). Für eine übergeordnete Erklärung, wie Quantenchemieprobleme auf Quantencomputer abgebildet werden, empfehlen wir die Vorlesung [*Mapping Problems to Qubits*](https://youtube.com/watch?v=TyFU6r8uEsE\&t=900) der Qiskit Global Summer School 2024.

### Lokaler unitärer Cluster-Jastrow-Ansatz (LUCJ)
SQD benötigt einen Quantenschaltkreisansatz, aus dem Stichproben gezogen werden können. In diesem Tutorial verwenden wir den [lokalen unitären Cluster-Jastrow-Ansatz (LUCJ)](https://pubs.rsc.org/en/content/articlelanding/2023/sc/d3sc02516k) aufgrund seiner Kombination aus physikalischer Motivation und Hardware-Freundlichkeit. Wir verwenden [ffsim](https://qiskit-community.github.io/ffsim/), um den Ansatzschaltkreis zu konstruieren.

Der LUCJ-Ansatz passt sich an QPUs mit eingeschränkter Qubit-Konnektivität an. Die Spinorbitale werden so auf Qubits abgebildet, dass der Ansatz kein Routing mit SWAP Gates erfordert. IBM&reg;-Hardware hat eine Heavy-Hex-Gitter-Qubit-Topologie, in diesem Fall können wir ein „Zickzack"-Muster verwenden, das unten dargestellt ist. In diesem Muster werden Orbitale mit demselben Spin auf Qubits mit einer Linientopologie abgebildet (rote und blaue Kreise), und eine Verbindung zwischen Orbitalen verschiedenen Spins besteht an jedem 4. räumlichen Orbital, wobei die Verbindung durch ein Ancilla-Qubit (lila Kreise) hergestellt wird.

![Qubit-Zuordnungsdiagramm für den LUCJ-Ansatz auf einem Heavy-Hex-Gitter](../docs/images/tutorials/improving-energy-estimation-of-a-fermionic-hamiltonian-with-sqd/7e0ee7e1-2d24-417f-ac59-25c58db79aa9.avif)

### Selbstkonsistente Konfigurationswiederherstellung
Das Verfahren der selbstkonsistenten Konfigurationswiederherstellung ist darauf ausgelegt, so viel Signal wie möglich aus verrauschten Quantenstichproben zu extrahieren. Da der molekulare Hamiltonoperator die Teilchenzahl und Spin-Z erhält, ist es sinnvoll, einen Schaltkreisansatz zu wählen, der diese Symmetrien ebenfalls erhält. Wenn er auf den Hartree-Fock-Zustand angewendet wird, hat der resultierende Zustand im rauschfreien Fall eine feste Teilchenzahl und Spin-Z. Daher sollte die Spin-$\alpha$- und Spin-$\beta$-Hälfte jedes Bitstrings, der aus diesem Zustand abgetastet wird, dasselbe [Hamming-Gewicht](https://en.wikipedia.org/wiki/Hamming_weight) wie im Hartree-Fock-Zustand haben. Aufgrund von Rauschen in aktuellen Quantenprozessoren verletzen einige gemessene Bitstrings diese Eigenschaft. Eine einfache Form der Nachselektion würde diese Bitstrings verwerfen, aber das ist verschwenderisch, da diese Bitstrings möglicherweise noch ein gewisses Signal enthalten. Das Verfahren der selbstkonsistenten Wiederherstellung versucht, einen Teil dieses Signals in der Nachbearbeitung wiederzugewinnen. Das Verfahren ist iterativ und benötigt als Eingabe eine Schätzung der mittleren Besetzungszahlen jedes Orbitals im Grundzustand, die zunächst aus den Rohstichproben berechnet wird. Das Verfahren wird in einer Schleife ausgeführt, und jede Iteration hat die folgenden Schritte:

1.  Für jeden Bitstring, der die angegebenen Symmetrien verletzt, werden seine Bits mit einem probabilistischen Verfahren umgekehrt, das darauf ausgelegt ist, den Bitstring näher an die aktuelle Schätzung der mittleren Orbitalbesetzungszahlen heranzubringen, um einen neuen Bitstring zu erhalten.
2.  Sammle alle alten und neuen Bitstrings, die die Symmetrien erfüllen, und nehme Teilmengen einer vorher festgelegten festen Größe als Teilstichproben.
3.  Projiziere für jede Teilmenge von Bitstrings den Hamiltonoperator in den durch die entsprechenden Basisvektoren aufgespannten Unterraum (siehe den [vorherigen Abschnitt](#quantum-chemistry) für eine Beschreibung dieser Basisvektoren) und berechne auf einem klassischen Computer eine Grundzustandsnäherung des projizierten Hamiltonoperators.
4.  Aktualisiere die Schätzung der mittleren Orbitalbesetzungszahlen mit der Grundzustandsnäherung mit der niedrigsten Energie.

### SQD-Workflow-Diagramm
Der SQD-Workflow ist im folgenden Diagramm dargestellt:

![Workflow-Diagramm des SQD-Algorithmus](../docs/images/tutorials/improving-energy-estimation-of-a-fermionic-hamiltonian-with-sqd/fd7e816f-4e2e-4dd7-a7da-f71afb9ca68d.avif)
## Anforderungen
Bevor du dieses Tutorial beginnst, stelle sicher, dass du Folgendes installiert hast:

*   Qiskit SDK v1.0 oder höher, mit Unterstützung für [Visualisierung](https://docs.quantum.ibm.com/api/qiskit/visualization)
*   Qiskit Runtime v0.22 oder höher (`pip install qiskit-ibm-runtime`)
*   SQD-Qiskit-Addon v0.11 oder höher (`pip install qiskit-addon-sqd`)
*   ffsim v0.0.75 oder höher (`pip install ffsim`)
## Einrichtung

In [1]:
import math

import ffsim
import matplotlib.pyplot as plt
import numpy as np
import pyscf
import pyscf.cc
import pyscf.mcscf
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.primitives import StatevectorSampler
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler

## Kleinskaliges Simulatorbeispiel

In diesem Tutorial werden wir eine Näherung an den Grundzustand eines Stickstoffmoleküls in der Nähe seines Gleichgewichtsbindungsabstands finden. Wir verwenden zunächst einen kleinen STO-6G-Basissatz, damit wir das Experiment simulieren und sicherstellen können, dass es funktioniert.

### Schritt 1: Klassische Eingaben auf ein Quantenproblem abbilden

Zuerst geben wir das Molekül und seine Eigenschaften an.

In [2]:
# Specify molecule properties
spin_sq = 0

# Build N2 molecule
mol = pyscf.gto.Mole()
mol.build(
    atom=[["N", (0, 0, 0)], ["N", (1.0, 0, 0)]],
    basis="sto-6g",
    symmetry="Dooh",
)

# Define active space
n_frozen = 2
active_space = range(n_frozen, mol.nao_nr())

# Get molecular integrals
scf = pyscf.scf.RHF(mol).run()
norb = len(active_space)
n_electrons = int(sum(scf.mo_occ[active_space]))
n_alpha = (n_electrons + mol.spin) // 2
n_beta = (n_electrons - mol.spin) // 2
nelec = (n_alpha, n_beta)
cas = pyscf.mcscf.CASCI(scf, norb, nelec)
mo = cas.sort_mo(active_space, base=0)
hcore, nuclear_repulsion_energy = cas.get_h1cas(mo)
eri = pyscf.ao2mo.restore(1, cas.get_h2cas(mo), norb)

# Compute exact energy using FCI
reference_energy = cas.run().e_tot

print(f"norb = {norb}")
print(f"nelec = {nelec}")

converged SCF energy = -108.464957764796
CASCI E = -108.595987350986  E(CI) = -32.4115475088426  S^2 = 0.0000000
norb = 8
nelec = (5, 5)


Before constructing the LUCJ ansatz circuit, we first perform a CCSD calculation in the following code cell. The [$t_1$ and $t_2$ amplitudes](https://en.wikipedia.org/wiki/Coupled_cluster#Cluster_operator) from this calculation will be used to initialize the parameters of the ansatz.

In [3]:
# Get CCSD t2 amplitudes for initializing the ansatz
ccsd = pyscf.cc.CCSD(
    scf, frozen=[i for i in range(mol.nao_nr()) if i not in active_space]
).run()
t1 = ccsd.t1
t2 = ccsd.t2

E(CCSD) = -108.5933309085008  E_corr = -0.1283731437052354


Bevor wir den LUCJ-Ansatzschaltkreis konstruieren, führen wir zunächst eine CCSD-Berechnung in der folgenden Codezelle durch. Die [$t_1$- und $t_2$-Amplituden](https://en.wikipedia.org/wiki/Coupled_cluster#Cluster_operator) aus dieser Berechnung werden verwendet, um die Parameter des Ansatzes zu initialisieren.

In [4]:
import warnings

from qiskit.transpiler import CouplingMap

warnings.formatwarning = lambda msg, *args, **kwargs: f"Warning: {msg}\n"

# Set ansatz properties
n_reps = 1
pairs_aa = [(p, p + 1) for p in range(norb - 1)]
pairs_ab = None  # Let generate_lucj_pass_manager determine the alpha-beta interactions

# Initialize backend
coupling_map = CouplingMap.from_heavy_hex(3)
backend = GenericBackendV2(
    coupling_map.size(),
    coupling_map=coupling_map,
    basis_gates=["cp", "xx_plus_yy", "p", "x", "swap"],
)

# Create pass manager
pass_manager, pairs_ab = ffsim.qiskit.generate_lucj_pass_manager(
    backend=backend,
    norb=norb,
    connectivity="heavy-hex",
    interaction_pairs=(pairs_aa, pairs_ab),
    optimization_level=3,
)

# Create the LUCJ ansatz operator
ucj_op = ffsim.UCJOpSpinBalanced.from_t_amplitudes(
    t2=t2,
    t1=t1,
    n_reps=n_reps,
    interaction_pairs=(pairs_aa, pairs_ab),
    # Setting optimize=True enables the "compressed" factorization
    optimize=True,
    # Limit the number of optimization iterations to prevent the code cell from running
    # too long. Removing this line may improve results.
    options=dict(maxiter=1000),
)

# create an empty quantum circuit
qubits = QuantumRegister(2 * norb, name="q")
circuit = QuantumCircuit(qubits)

# prepare Hartree-Fock state as the reference state and append it to the quantum circuit
circuit.append(ffsim.qiskit.PrepareHartreeFockJW(norb, nelec), qubits)

# apply the UCJ operator to the reference state
circuit.append(ffsim.qiskit.UCJOpSpinBalancedJW(ucj_op), qubits)
circuit.measure_all()

### Step 2: Optimize for quantum hardware execution

Next, we optimize the circuit for a target hardware. Typically, this step involves initializing the hardware backend and a pass manager for that backend. However, since the LUCJ ansatz is adapted to the hardware connectivity, we already performed these actions in the previous step. All that is left to do is run the pass manager on the circuit to transpile it to an ISA circuit that can be directly executed on the QPU.

In [5]:
isa_circuit = pass_manager.run(circuit)
print(f"Gate counts: {isa_circuit.count_ops()}")

Gate counts: OrderedDict({'xx_plus_yy': 86, 'p': 16, 'measure': 16, 'cp': 15, 'x': 10, 'swap': 2, 'barrier': 1})


Nun verwenden wir [ffsim](https://github.com/qiskit-community/ffsim), um den Ansatzschaltkreis zu erstellen. Da unser Molekül einen geschlossenschaligen Hartree-Fock-Zustand hat, verwenden wir die spinbalancierte Variante des UCJ-Ansatzes, [UCJOpSpinBalanced](https://qiskit-community.github.io/ffsim/api/ffsim.html#ffsim.UCJOpSpinBalanced). Wir setzen `optimize=True` in der `from_t_amplitudes`-Methode, um die „komprimierte" Doppelfaktorisierung der $t_2$-Amplituden zu aktivieren (siehe [The local unitary cluster Jastrow (LUCJ) ansatz](https://qiskit-community.github.io/ffsim/explanations/lucj.html#Parameter-initialization-from-CCSD) in der ffsim-Dokumentation für Details).

Da der LUCJ-Ansatz sich an die verfügbare Konnektivität der QPU anpasst, müssen wir das QPU-Backend initialisieren, bevor wir den Ansatz erstellen. Zunächst erstellen wir ein generisches Backend mit einer Heavy-Hex-Kopplungskarte und einem Gate-Satz, in den sich der LUCJ-Ansatz natürlich zerlegen lässt. Dann verwenden wir `ffsim.qiskit.generate_lucj_pass_manager`, um einen Pass-Manager zu erstellen, der spezialisiert ist auf die Transpilierung des LUCJ-Ansatzes auf das gegebene Backend gemäß dem im [Hintergrundabschnitt zum LUCJ-Ansatz](#local-unitary-cluster-jastrow-lucj-ansatz) beschriebenen „Zickzack"-Layout. Diese Funktion verwendet eine Bewertungsheuristik, um die mit dem ausgewählten Layout verbundenen Fehler zu minimieren, was wichtig ist, wenn dein Backend eine echte QPU oder ein Simulator mit einem Rauschmodell ist. Neben der Rückgabe des Pass-Managers gibt diese Funktion auch die Alpha-Beta-Kopplungspaare zurück, die auf der Hardware implementiert werden können. Wenn nicht alle Paare implementiert werden können, gibt sie eine Warnung aus.

In [6]:
rng = np.random.default_rng()
sampler = StatevectorSampler(seed=rng)
job = sampler.run([isa_circuit], shots=100_000)

In [7]:
primitive_result = job.result()
pub_result = primitive_result[0]

### Step 4: Post-process and return result in desired classical format

A useful metric to judge the quality of the QPU output is the number of valid configurations returned. A valid configuration has the correct particle number and spin Z, which means that the right half of the bitstring has Hamming weight equal to the number of spin-up electrons, and the left half has Hamming weight equal to the number of spin-down electrons. The following cell computes the fraction of sampled configurations that are valid.

In [8]:
def is_valid_bitstring(
    bitstring: str, norb: int, nelec: tuple[int, int]
) -> bool:
    n_alpha, n_beta = nelec
    return (
        len(bitstring) == 2 * norb
        and bitstring[norb:].count("1") == n_alpha
        and bitstring[:norb].count("1") == n_beta
    )


bit_array = pub_result.data.meas
num_valid = sum(
    is_valid_bitstring(b, norb, nelec) for b in bit_array.get_bitstrings()
)
valid_fraction = num_valid / bit_array.num_shots
print(f"Fraction of sampled configurations that are valid: {valid_fraction}")

Fraction of sampled configurations that are valid: 1.0


### Schritt 3: Mit Qiskit-Primitives ausführen
Nachdem wir den Circuit für die Hardware-Ausführung optimiert haben, sind wir bereit, ihn auf der Zielhardware auszuführen und Stichproben für die Grundzustandsenergieabschätzung zu sammeln. Da wir nur einen Circuit haben, werden wir den [Job-Ausführungsmodus](/guides/execution-modes) von Qiskit Runtime verwenden und unseren Circuit ausführen.

In [9]:
expected_fraction_random = (
    math.comb(norb, n_alpha) * math.comb(norb, n_beta) / 2 ** (2 * norb)
)
print(
    f"Expected fraction of valid configurations from uniformly random bitstrings: {expected_fraction_random}"
)

Expected fraction of valid configurations from uniformly random bitstrings: 0.0478515625


Now, we estimate the ground state energy of the Hamiltonian using the `diagonalize_fermionic_hamiltonian` function. This function performs the self-consistent configuration recovery procedure to iteratively refine the noisy quantum samples to improve the energy estimate. We pass a callback function so that we can save the intermediate results for later analysis. See the [API documentation](/docs/api/qiskit-addon-sqd/fermion#diagonalize_fermionic_hamiltonian) for explanations of the arguments to `diagonalize_fermionic_hamiltonian`.

Here, we use the `initial_occupancies` argument to `diagonalize_fermionic_hamiltonian` to specify the Hartree-Fock configuration as the initial guess for the orbital occupancies in the ground state. This approach is sensible for systems where the ground state has significant support on the Hartree-Fock configuration, but it might not be appropriate in other situations, though more advanced computational methods might yield better initial guesses in those cases. Specifying `initial_occupancies` also allows configuration recovery to run even if no valid configurations were sampled, as may be the case when sampling a large circuit on a noisy QPU. Without this argument, the configuration recovery would fail and raise an error if no valid configurations were provided.

In [10]:
from functools import partial

from qiskit_addon_sqd.fermion import (
    SCIResult,
    diagonalize_fermionic_hamiltonian,
    solve_sci_batch,
)

# SQD options
energy_tol = 1e-3
occupancies_tol = 1e-3
max_iterations = 5

# Eigenstate solver options
num_batches = 3
samples_per_batch = 300
symmetrize_spin = True
carryover_threshold = 1e-4
max_cycle = 200

# Use the Hartree-Fock configuration as an initial guess for the orbital occupancies
initial_occupancies = (
    np.array([1] * n_alpha + [0] * (norb - n_alpha)),
    np.array([1] * n_beta + [0] * (norb - n_beta)),
)

# Pass options to the built-in eigensolver. If you just want to use the defaults,
# you can omit this step, in which case you would not specify the sci_solver argument
# in the call to diagonalize_fermionic_hamiltonian below.
sci_solver = partial(solve_sci_batch, spin_sq=0.0, max_cycle=max_cycle)

# List to capture intermediate results
result_history = []


def callback(results: list[SCIResult]):
    result_history.append(results)
    iteration = len(result_history)
    print(f"Iteration {iteration}")
    for i, result in enumerate(results):
        print(f"\tSubsample {i}")
        print(f"\t\tEnergy: {result.energy + nuclear_repulsion_energy}")
        print(
            f"\t\tSubspace dimension: {np.prod(result.sci_state.amplitudes.shape)}"
        )


result = diagonalize_fermionic_hamiltonian(
    hcore,
    eri,
    bit_array,
    samples_per_batch=samples_per_batch,
    norb=norb,
    nelec=nelec,
    num_batches=num_batches,
    energy_tol=energy_tol,
    occupancies_tol=occupancies_tol,
    max_iterations=max_iterations,
    sci_solver=sci_solver,
    symmetrize_spin=symmetrize_spin,
    initial_occupancies=initial_occupancies,
    carryover_threshold=carryover_threshold,
    callback=callback,
    seed=rng,
)

final_energy = result.energy + nuclear_repulsion_energy
energy_error = final_energy - reference_energy
print(f"Final energy: {final_energy}")
print(f"Final energy error: {energy_error}")

Iteration 1
	Subsample 0
		Energy: -108.59275573641656
		Subspace dimension: 900
	Subsample 1
		Energy: -108.59275573641656
		Subspace dimension: 900
	Subsample 2
		Energy: -108.59275573641656
		Subspace dimension: 900
Iteration 2
	Subsample 0
		Energy: -108.59275573641656
		Subspace dimension: 900
	Subsample 1
		Energy: -108.59275573641656
		Subspace dimension: 900
	Subsample 2
		Energy: -108.59275573641656
		Subspace dimension: 900
Final energy: -108.59275573641656
Final energy error: 0.0032316145694579745


#### Visualize the results

The first plot shows that in this simulation we are already within `1 mH` of the exact answer after the first iteration (chemical accuracy is typically accepted to be `1 kcal/mol` $\approx$ `1.6 mH`). This is a small system, though, and because the samples are noiseless, configuration recovery is not needed. On a larger system run on a noisy QPU, multiple configuration recovery iterations might be needed, and the final accuracy might be worse. Generally, the energy can be improved by allowing more configuration recovery iterations or by increasing the number of samples per batch.

The second plot shows the average occupancy of each spatial orbital after the final iteration. We can see that both the spin-up and spin-down electrons occupy the first five orbitals with high probability in our solutions.

In [11]:
# Data for energies plot
x1 = range(len(result_history))
min_e = [
    min(result, key=lambda res: res.energy).energy + nuclear_repulsion_energy
    for result in result_history
]
e_diff = [abs(e - reference_energy) for e in min_e]
yt1 = [1.0, 1e-1, 1e-2, 1e-3, 1e-4]

# Chemical accuracy (+/- 1 milli-Hartree)
chem_accuracy = 0.001

# Data for avg spatial orbital occupancy
y2 = np.sum(result.orbital_occupancies, axis=0)
x2 = range(len(y2))

fig, axs = plt.subplots(1, 2, figsize=(12, 6))

# Plot energies
axs[0].plot(x1, e_diff, label="energy error", marker="o")
axs[0].set_xticks(x1)
axs[0].set_xticklabels(x1)
axs[0].set_yticks(yt1)
axs[0].set_yticklabels(yt1)
axs[0].set_yscale("log")
axs[0].set_ylim(1e-4)
axs[0].axhline(
    y=chem_accuracy,
    color="#BF5700",
    linestyle="--",
    label="chemical accuracy",
)
axs[0].set_title("Approximated Ground State Energy Error vs SQD Iterations")
axs[0].set_xlabel("Iteration Index", fontdict={"fontsize": 12})
axs[0].set_ylabel("Energy Error (Ha)", fontdict={"fontsize": 12})
axs[0].legend()

# Plot orbital occupancy
axs[1].bar(x2, y2, width=0.8)
axs[1].set_xticks(x2)
axs[1].set_xticklabels(x2)
axs[1].set_title("Avg Occupancy per Spatial Orbital")
axs[1].set_xlabel("Orbital Index", fontdict={"fontsize": 12})
axs[1].set_ylabel("Avg Occupancy", fontdict={"fontsize": 12})

plt.tight_layout()
plt.show()

<Image src="../docs/images/tutorials/sample-based-quantum-diagonalization/extracted-outputs/caffd888-e89c-4aa9-8bae-4d1bb723b35e-0.avif" alt="Output of the previous code cell" />

### Schritt 4: Nachbearbeiten und Ergebnis im gewünschten klassischen Format zurückgeben
Eine nützliche Kenngröße zur Beurteilung der Qualität der QPU-Ausgabe ist die Anzahl der zurückgegebenen gültigen Konfigurationen. Eine gültige Konfiguration hat die korrekte Teilchenzahl und Spin-Z, was bedeutet, dass die rechte Hälfte des Bitstrings das Hamming-Gewicht gleich der Anzahl der Spin-up-Elektronen hat und die linke Hälfte das Hamming-Gewicht gleich der Anzahl der Spin-down-Elektronen. Die folgende Zelle berechnet den Anteil der abgetasteten Konfigurationen, die gültig sind.

In [12]:
# ------------------------------ Step 1 ------------------------------
# Build N2 molecule
mol = pyscf.gto.Mole()
mol.build(
    atom=[["N", (0, 0, 0)], ["N", (1.0, 0, 0)]],
    basis="cc-pvdz",
    symmetry="Dooh",
)

# Define active space
n_frozen = 2
active_space = range(n_frozen, mol.nao_nr())

# Get molecular integrals
scf = pyscf.scf.RHF(mol).run()
norb = len(active_space)
n_electrons = int(sum(scf.mo_occ[active_space]))
n_alpha = (n_electrons + mol.spin) // 2
n_beta = (n_electrons - mol.spin) // 2
nelec = (n_alpha, n_beta)
cas = pyscf.mcscf.CASCI(scf, norb, nelec)
mo = cas.sort_mo(active_space, base=0)
hcore, nuclear_repulsion_energy = cas.get_h1cas(mo)
eri = pyscf.ao2mo.restore(1, cas.get_h2cas(mo), norb)

# Store reference energy from SCI calculation performed separately
reference_energy = -109.22802921665716

print(f"norb = {norb}")
print(f"nelec = {nelec}")

# Get CCSD t2 amplitudes for initializing the ansatz
ccsd = pyscf.cc.CCSD(
    scf, frozen=[i for i in range(mol.nao_nr()) if i not in active_space]
).run()
t1 = ccsd.t1
t2 = ccsd.t2

# Set ansatz properties
n_reps = 1
pairs_aa = [(p, p + 1) for p in range(norb - 1)]
pairs_ab = None  # Let generate_lucj_pass_manager determine the alpha-beta interactions

# Initialize backend
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=133
)
print(f"Using backend {backend.name}")

# Create pass manager
pass_manager, pairs_ab = ffsim.qiskit.generate_lucj_pass_manager(
    backend=backend,
    norb=norb,
    connectivity="heavy-hex",
    interaction_pairs=(pairs_aa, pairs_ab),
    optimization_level=3,
)

# Create the LUCJ ansatz operator
ucj_op = ffsim.UCJOpSpinBalanced.from_t_amplitudes(
    t2=t2,
    t1=t1,
    n_reps=n_reps,
    interaction_pairs=(pairs_aa, pairs_ab),
    # Setting optimize=True enables the "compressed" factorization
    optimize=True,
    # Limit the number of optimization iterations to prevent the code cell from running
    # too long. Removing this line may improve results.
    options=dict(maxiter=1000),
)

# create an empty quantum circuit
qubits = QuantumRegister(2 * norb, name="q")
circuit = QuantumCircuit(qubits)

# prepare Hartree-Fock state as the reference state and append it to the quantum circuit
circuit.append(ffsim.qiskit.PrepareHartreeFockJW(norb, nelec), qubits)

# apply the UCJ operator to the reference state
circuit.append(ffsim.qiskit.UCJOpSpinBalancedJW(ucj_op), qubits)
circuit.measure_all()


# ------------------------------ Step 2 ------------------------------

isa_circuit = pass_manager.run(circuit)
print(f"Gate counts: {isa_circuit.count_ops()}")


# ------------------------------ Step 3 ------------------------------
sampler = Sampler(mode=backend)
sampler.options.environment.job_tags = ["TUT_SQD"]
job = sampler.run([isa_circuit], shots=100_000)
primitive_result = job.result()
pub_result = primitive_result[0]


# ------------------------------ Step 4 ------------------------------

bit_array = pub_result.data.meas
num_valid = sum(
    is_valid_bitstring(b, norb, nelec) for b in bit_array.get_bitstrings()
)
valid_fraction = num_valid / bit_array.num_shots
print(f"Fraction of sampled configurations that are valid: {valid_fraction}")
expected_fraction_random = (
    math.comb(norb, n_alpha) * math.comb(norb, n_beta) / 2 ** (2 * norb)
)
print(
    f"Expected fraction of valid configurations from uniformly random bitstrings: {expected_fraction_random}"
)
# SQD options
energy_tol = 1e-3
occupancies_tol = 1e-3
max_iterations = 5

# Eigenstate solver options
num_batches = 3
samples_per_batch = 300
symmetrize_spin = True
carryover_threshold = 1e-4
max_cycle = 200

# Use the Hartree-Fock configuration as an initial guess for the orbital occupancies
initial_occupancies = (
    np.array([1] * n_alpha + [0] * (norb - n_alpha)),
    np.array([1] * n_beta + [0] * (norb - n_beta)),
)

# Pass options to the built-in eigensolver. If you just want to use the defaults,
# you can omit this step, in which case you would not specify the sci_solver argument
# in the call to diagonalize_fermionic_hamiltonian below.
sci_solver = partial(solve_sci_batch, spin_sq=0.0, max_cycle=max_cycle)

# List to capture intermediate results
result_history = []


result = diagonalize_fermionic_hamiltonian(
    hcore,
    eri,
    bit_array,
    samples_per_batch=samples_per_batch,
    norb=norb,
    nelec=nelec,
    num_batches=num_batches,
    energy_tol=energy_tol,
    occupancies_tol=occupancies_tol,
    max_iterations=max_iterations,
    sci_solver=sci_solver,
    symmetrize_spin=symmetrize_spin,
    initial_occupancies=initial_occupancies,
    carryover_threshold=carryover_threshold,
    callback=callback,
    seed=rng,
)

final_energy = result.energy + nuclear_repulsion_energy
energy_error = final_energy - reference_energy
print(f"Final energy: {final_energy}")
print(f"Final energy error: {energy_error}")

# Data for energies plot
x1 = range(len(result_history))
min_e = [
    min(result, key=lambda res: res.energy).energy + nuclear_repulsion_energy
    for result in result_history
]
e_diff = [abs(e - reference_energy) for e in min_e]
yt1 = [1.0, 1e-1, 1e-2, 1e-3, 1e-4]

# Chemical accuracy (+/- 1 milli-Hartree)
chem_accuracy = 0.001

# Data for avg spatial orbital occupancy
y2 = np.sum(result.orbital_occupancies, axis=0)
x2 = range(len(y2))

fig, axs = plt.subplots(1, 2, figsize=(12, 6))

# Plot energies
axs[0].plot(x1, e_diff, label="energy error", marker="o")
axs[0].set_xticks(x1)
axs[0].set_xticklabels(x1)
axs[0].set_yticks(yt1)
axs[0].set_yticklabels(yt1)
axs[0].set_yscale("log")
axs[0].set_ylim(1e-4)
axs[0].axhline(
    y=chem_accuracy,
    color="#BF5700",
    linestyle="--",
    label="chemical accuracy",
)
axs[0].set_title("Approximated Ground State Energy Error vs SQD Iterations")
axs[0].set_xlabel("Iteration Index", fontdict={"fontsize": 12})
axs[0].set_ylabel("Energy Error (Ha)", fontdict={"fontsize": 12})
axs[0].legend()

# Plot orbital occupancy
axs[1].bar(x2, y2, width=0.8)
axs[1].set_xticks(x2)
axs[1].set_xticklabels(x2)
axs[1].set_title("Avg Occupancy per Spatial Orbital")
axs[1].set_xlabel("Orbital Index", fontdict={"fontsize": 12})
axs[1].set_ylabel("Avg Occupancy", fontdict={"fontsize": 12})

plt.tight_layout()
plt.show()

converged SCF energy = -108.929838385609
norb = 26
nelec = (5, 5)
E(CCSD) = -109.2177884185544  E_corr = -0.2879500329450045
Using backend ibm_boston


Removing interaction (24, 24) from the end.
Removing interaction (20, 20) from the end.


Gate counts: OrderedDict({'sx': 7039, 'rz': 6990, 'cz': 1858, 'x': 61, 'measure': 52, 'barrier': 1})
Fraction of sampled configurations that are valid: 0.02124
Expected fraction of valid configurations from uniformly random bitstrings: 9.607888706852918e-07
Iteration 1
	Subsample 0
		Energy: -109.13889134249762
		Subspace dimension: 120409
	Subsample 1
		Energy: -109.11785470455858
		Subspace dimension: 110889
	Subsample 2
		Energy: -109.13234360554011
		Subspace dimension: 130321
Iteration 2
	Subsample 0
		Energy: -109.16392179579177
		Subspace dimension: 223729
	Subsample 1
		Energy: -109.16281938332986
		Subspace dimension: 223729
	Subsample 2
		Energy: -109.16955816711932
		Subspace dimension: 233289
Iteration 3
	Subsample 0
		Energy: -109.17905772999075
		Subspace dimension: 324900
	Subsample 1
		Energy: -109.17532445048462
		Subspace dimension: 357604
	Subsample 2
		Energy: -109.1733168689756
		Subspace dimension: 348100
Iteration 4
	Subsample 0
		Energy: -109.18437778820451
		Su

<Image src="../docs/images/tutorials/sample-based-quantum-diagonalization/extracted-outputs/3858949c-a55d-4ff8-a0fc-54fb53e131b5-3.avif" alt="Output of the previous code cell" />

## Next steps

<Admonition type="tip" title="Recommendations">
If you found this work interesting, you might be interested in the following material:
- [Sample-based Krylov quantum diagonalization of a fermionic lattice model](/docs/tutorials/sample-based-krylov-quantum-diagonalization) - a related tutorial using time evolution circuits instead of a variational ansatz
- [Scale SQD chemistry workflows with Dice solver](https://qiskit.github.io/qiskit-addon-sqd/how_tos/integrate_dice_solver.html) - a page showing how to use the more efficient Dice software for diagonalization
- [SQD addon API documentation](https://qiskit.github.io/qiskit-addon-sqd/apidocs/qiskit_addon_sqd.fermion.html#qiskit_addon_sqd.fermion.diagonalize_fermionic_hamiltonian) - reference for the `diagonalize_fermionic_hamiltonian` function
- [*Chemistry beyond the scale of exact diagonalization on a quantum-centric supercomputer*](https://www.science.org/doi/10.1126/sciadv.adu9991) - the paper this tutorial is based on
</Admonition>